# Contextual Multi-Armed Bandit

For the contextual multi-armed bandit (cMAB) when user information is available (context), we implemented a generalisation of Thompson sampling algorithm ([Agrawal and Goyal, 2014](https://arxiv.org/pdf/1209.3352.pdf)) based on NumPyro.

![title](img/cmab.png)

The following notebook contains an example of usage of the class Cmab, which implements the algorithm above.

In [1]:
import numpy as np

from pybandits.cmab import CmabBernoulli
from pybandits.model import BayesianNeuralNetwork, BnnLayerParams, BnnParams, FeaturesConfig, StudentTArray

/home/runner/.cache/pypoetry/virtualenvs/pybandits-vYJB-miV-py3.10/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
n_samples = 1000
n_features = 5

First, we need to define the input context matrix $X$ of size ($n\_samples, n\_features$) and the mapping of possible actions $a_i \in A$ to their associated model.

In [3]:
# context
X = 2 * np.random.random_sample((n_samples, n_features)) - 1  # random float in the interval (-1, 1)
print("X: context matrix of shape (n_samples, n_features)")
print(X[:10])

X: context matrix of shape (n_samples, n_features)
[[-0.61336456  0.41217676  0.84297469  0.57590829  0.48771431]
 [-0.46213049  0.11738824  0.89234046  0.96737041 -0.25355038]
 [-0.09495053  0.2201867   0.31763381 -0.21477589  0.55420897]
 [-0.83961856 -0.48843947  0.27154354  0.24356162  0.54908941]
 [ 0.45391837  0.7148274  -0.95039011 -0.97447896  0.03756093]
 [ 0.12389446  0.41092807 -0.36309819  0.9630806   0.02469044]
 [-0.69241402 -0.85950555 -0.3681327   0.72000942  0.56116047]
 [ 0.91438486  0.26661237  0.4227113  -0.24458061  0.27766863]
 [ 0.4452133  -0.20081204 -0.91487865  0.46905418 -0.38883758]
 [-0.87853823  0.82538197  0.29321064 -0.69855592  0.5230764 ]]


In [4]:
# define action model
bias = StudentTArray.cold_start(mu=1, sigma=2, shape=1)
weight = StudentTArray.cold_start(shape=(n_features, 1))
layer_params = BnnLayerParams(weight=weight, bias=bias)
model_params = BnnParams(bnn_layer_params=[layer_params])
feature_config = FeaturesConfig(n_features=n_features)

update_kwargs = {"num_steps": 100, "batch_size": 128, "optimizer_type": "adam"}

actions = {
    "a1": BayesianNeuralNetwork(
        model_params=model_params,
        feature_config=feature_config,
        update_kwargs=update_kwargs,
    ),
    "a2": BayesianNeuralNetwork(
        model_params=model_params,
        feature_config=feature_config,
        update_kwargs=update_kwargs,
    ),
}

We can now init the bandit given the mapping of actions $a_i$ to their model.

In [5]:
# init contextual Multi-Armed Bandit model
cmab = CmabBernoulli(actions=actions)

The predict function below returns the action selected by the bandit at time $t$: $a_t = argmax_k P(r=1|\beta_k, x_t)$. The bandit selects one action per each sample of the contect matrix $X$.

In [6]:
# predict action
pred_actions, _, _ = cmab.predict(X)
print("Recommended action: {}".format(pred_actions[:10]))

Recommended action: ['a2', 'a2', 'a2', 'a1', 'a1', 'a1', 'a1', 'a1', 'a2', 'a2']


Now, we observe the rewards and the context from the environment. In this example rewards and the context are randomly simulated.

In [7]:
# simulate reward from environment
simulated_rewards = np.random.randint(2, size=n_samples).tolist()
print("Simulated rewards: {}".format(simulated_rewards[:10]))

Simulated rewards: [1, 0, 0, 0, 1, 1, 0, 0, 0, 1]


Finally, we update the model providing per each action sample: (i) its context $x_t$ (ii) the action $a_t$ selected by the bandit, (iii) the corresponding reward $r_t$.

In [8]:
# update model
cmab.update(context=X, actions=pred_actions, rewards=simulated_rewards)

SVI:   0%|          | 0/25 [00:00<?, ?it/s]

SVI:   4%|▍         | 1/25 [00:01<00:25,  1.06s/it]

SVI:   4%|▍         | 1/25 [00:01<00:25,  1.06s/it, loss=2564.5994]

SVI:   8%|▊         | 2/25 [00:01<00:24,  1.06s/it, loss=2735.7546]

SVI:  12%|█▏        | 3/25 [00:01<00:23,  1.06s/it, loss=3049.0017]

...

SVI:  91%|█████████ | 31/34 [00:00<00:02,  1.10it/s, loss=3662.6865]

SVI:  94%|█████████▍| 32/34 [00:00<00:01,  1.10it/s, loss=2352.4646]

SVI:  97%|█████████▋| 33/34 [00:00<00:00,  1.10it/s, loss=2437.1094]

SVI: 100%|██████████| 34/34 [00:01<00:00, 22.66it/s, loss=2437.1094]

SVI: 100%|██████████| 34/34 [00:01<00:00, 22.66it/s, loss=1930.9392]